# 第2章 预备知识 · 2.5 自动求导（autograd）

深度学习训练的核心：你只写前向，框架自动算梯度。本笔记覆盖 `requires_grad`、`backward`、非标量求导与 `detach`。

In [1]:
import torch

x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [2]:
x.requires_grad_(True)  # 等价于x=torch.arange(4.0,requires_grad=True)
x.grad  # 默认值是None

## 2. 标量反向传播（backward）

`requires_grad_(True)` 开启追踪；算完 `y=2·x·x` 后 `y.backward()` 自动得到 `x.grad=4x`。**注意**：`x.grad` 默认会累积，每次新计算前要 `zero_()` 清空。

In [3]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [24]:
y.backward()
x.grad

tensor([2., 2., 2., 2.])

In [5]:
x.grad == 4 * x

tensor([True, True, True, True])

In [23]:
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

## 3. 非标量的 backward

向量不能直接 `backward()`，要先 `sum()` 或传 `torch.ones` 当上游梯度——本质就是"把各分量的偏导求和"。

In [7]:
# 对非标量调用backward需要传入一个gradient参数，该参数指定微分函数关于self的梯度。
# 本例只想求偏导数的和，所以传递一个1的梯度是合适的
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

## 4. detach：从计算图剪断（当成常数）

`y.detach()` 把张量当常数，梯度传到它就断。于是 `z=detach(y)·x` 求导只剩 `∂z/∂x=u`，得到 `x.grad==u`。

In [8]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [9]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

## 5. 含控制流的函数也能求导（动态图）

函数里有 `while` / `if`，backward 照样能求导：它记录"实际走过的运算路径"，再沿链式法则回放。本例中 `d` 与 `a` 成正比，故 `a.grad == d/a`。

In [10]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

In [11]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

In [12]:
a.grad == d / a

tensor(True)